# Modeling Backtest — Revised Non-overlapping PnL

## Notebook目的

将未来3秒预测转化为可审计的持仓过程，同时用不重叠的500ms增量收益记账，避免把每500ms生成的重叠3秒标签连续复利。

## 输入数据

- Validation阶段：`Validation_data_revised.pkl`与`validation_predictions_revised.pkl`
- 最终测试阶段：`Test_data_revised.pkl`与`test_predictions_revised.pkl`

## 输出结果

运行后可生成：

- 逐步信号、仓位、执行价格、成本和PnL
- 模型回测指标表
- Validation阶段冻结的信号阈值说明

当前Notebook不包含虚构结果。

## 对应研究文档

- [[../../01_Research/技术路线|技术路线]]
- [[../../04_Model/模型评价|模型评价]]
- [[../../06_Experiment/experiment_log|Experiment Log]]
- [[../../07_Report/项目总结报告|项目总结报告]]


## 时间线定义

```text
t：读取截至t的盘口并生成预测
t+500ms：按下一快照的ask/bid近似执行，形成新仓位
之后：每500ms使用非重叠mid-price增量计算持仓PnL
信号变化：在下一快照调仓
交易时段结束：强制平仓
```

未来3秒收益只作为模型预测目标。实际策略PnL按连续、不重叠的500ms持仓收益累计。


In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "PROJECT_STATUS.md").exists():
            return candidate
    raise FileNotFoundError("无法定位项目根目录。")

PROJECT_ROOT = (
    Path(os.environ["ML_FUTURES_PROJECT_ROOT"]).resolve()
    if "ML_FUTURES_PROJECT_ROOT" in os.environ
    else find_project_root()
)
DATA_DIR = PROJECT_ROOT / "02_Data/processed/revised"
MODEL_ARTIFACT_DIR = PROJECT_ROOT / "05_Code/revised/artifacts"
OUTPUT_DIR = PROJECT_ROOT / "05_Code/revised/artifacts/backtest"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODE = "validation"  # 最终模型冻结后才改为 "test"
MODEL_NAME = "lightgbm"
PREDICTION_COLUMN = f"pred_{MODEL_NAME}"

# Validation阶段按预先声明的活跃度规则确定阈值；
# Test阶段必须填写已经在Validation冻结的具体阈值。
SIGNAL_ACTIVE_QUANTILE = 0.90
FROZEN_TEST_THRESHOLD = None

FEE_RATE = 3e-6
EXTRA_SLIPPAGE_RATE = 3e-5
DAYS_PER_YEAR = 252


In [ ]:
if MODE == "validation":
    data_path = DATA_DIR / "Validation_data_revised.pkl"
    prediction_path = (
        MODEL_ARTIFACT_DIR / "validation_predictions_revised.pkl"
    )
elif MODE == "test":
    data_path = DATA_DIR / "Test_data_revised.pkl"
    prediction_path = (
        MODEL_ARTIFACT_DIR / "test_predictions_revised.pkl"
    )
else:
    raise ValueError("MODE只能是validation或test。")

if not data_path.exists() or not prediction_path.exists():
    raise FileNotFoundError(
        "请按顺序运行revised数据Notebook和模型Notebook。"
    )

market_data = pd.read_pickle(data_path).sort_values("trade_time")
predictions = pd.read_pickle(prediction_path).sort_values("trade_time")

if PREDICTION_COLUMN not in predictions.columns:
    raise KeyError(
        f"预测文件中不存在 {PREDICTION_COLUMN}。"
    )

prediction_keys = ["trade_time", "trade_date", "session_id"]
backtest_input = market_data[
    [
        "trade_time", "trade_date", "session_id",
        "pb1", "pa1", "mid_price", "last_price",
        "target_return_3s",
    ]
].merge(
    predictions[prediction_keys + [PREDICTION_COLUMN]],
    on=prediction_keys,
    how="inner",
    validate="one_to_one",
)
if len(backtest_input) != len(market_data):
    raise ValueError(
        "行情与预测未能一一匹配，请检查数据版本和时间键。"
    )
backtest_input.sort_values("trade_time", inplace=True)

if MODE == "validation":
    signal_threshold = float(
        backtest_input[PREDICTION_COLUMN]
        .abs()
        .quantile(SIGNAL_ACTIVE_QUANTILE)
    )
else:
    if FROZEN_TEST_THRESHOLD is None:
        raise ValueError(
            "Test回测前必须填写从Validation冻结的阈值。"
        )
    signal_threshold = float(FROZEN_TEST_THRESHOLD)

signal_threshold


In [ ]:
def run_incremental_backtest(
    frame,
    prediction_column,
    threshold,
    fee_rate,
    extra_slippage_rate,
):
    result_groups = []

    for _, session in frame.groupby(
        ["trade_date", "session_id"], sort=True
    ):
        result = session.sort_values("trade_time").copy()

        result["signal"] = np.select(
            [
                result[prediction_column] > threshold,
                result[prediction_column] < -threshold,
            ],
            [1.0, -1.0],
            default=0.0,
        )

        # t时刻信号在下一条500ms快照执行。
        result["position"] = result["signal"].shift(1).fillna(0.0)
        # 每个交易时段结束前强制回到空仓。
        result.iloc[-1, result.columns.get_loc("position")] = 0.0

        result["position_change"] = (
            result["position"].diff().fillna(result["position"]).abs()
        )
        signed_change = (
            result["position"].diff().fillna(result["position"])
        )
        result["execution_price"] = np.select(
            [signed_change > 0, signed_change < 0],
            [result["pa1"], result["pb1"]],
            default=np.nan,
        )

        result["next_mid_price"] = result["mid_price"].shift(-1)
        result["incremental_mid_return"] = (
            result["next_mid_price"] / result["mid_price"] - 1
        ).fillna(0.0)
        result["gross_strategy_return"] = (
            result["position"] * result["incremental_mid_return"]
        )

        half_spread_rate = (
            (result["pa1"] - result["pb1"])
            / (2 * result["mid_price"])
        )
        result["spread_cost"] = (
            result["position_change"] * half_spread_rate
        )
        result["fee_cost"] = (
            result["position_change"] * fee_rate
        )
        result["slippage_cost"] = (
            result["position_change"] * extra_slippage_rate
        )
        result["transaction_cost"] = (
            result["spread_cost"]
            + result["fee_cost"]
            + result["slippage_cost"]
        )
        result["net_strategy_return"] = (
            result["gross_strategy_return"]
            - result["transaction_cost"]
        )
        result_groups.append(result)

    result = pd.concat(result_groups).sort_values("trade_time")
    result["equity_curve"] = (
        1 + result["net_strategy_return"]
    ).cumprod()
    result["running_max"] = result["equity_curve"].cummax()
    result["drawdown"] = (
        result["equity_curve"] / result["running_max"] - 1
    )
    return result

backtest_result = run_incremental_backtest(
    backtest_input,
    PREDICTION_COLUMN,
    signal_threshold,
    FEE_RATE,
    EXTRA_SLIPPAGE_RATE,
)


In [ ]:
def summarize_backtest(result, days_per_year=252):
    daily_returns = result.groupby("trade_date")[
        "net_strategy_return"
    ].apply(lambda values: (1 + values).prod() - 1)

    daily_std = daily_returns.std(ddof=1)
    annualized_sharpe = (
        np.sqrt(days_per_year)
        * daily_returns.mean()
        / daily_std
        if len(daily_returns) >= 2 and daily_std > 0
        else np.nan
    )

    return {
        "total_return": float(result["equity_curve"].iloc[-1] - 1),
        "annualized_sharpe_from_daily_returns": float(
            annualized_sharpe
        ),
        "max_drawdown": float(result["drawdown"].min()),
        "turnover": float(result["position_change"].sum()),
        "position_change_count": int(
            result["position_change"].gt(0).sum()
        ),
        "active_position_ratio": float(
            result["position"].ne(0).mean()
        ),
        "trade_day_count": int(len(daily_returns)),
        "signal_threshold": float(signal_threshold),
        "fee_rate": float(FEE_RATE),
        "extra_slippage_rate": float(EXTRA_SLIPPAGE_RATE),
        "mode": MODE,
        "model": MODEL_NAME,
    }

backtest_metrics = summarize_backtest(backtest_result)

pd.DataFrame([backtest_metrics]).to_csv(
    OUTPUT_DIR / f"{MODE}_{MODEL_NAME}_metrics_revised.csv",
    index=False,
)
backtest_result.to_pickle(
    OUTPUT_DIR / f"{MODE}_{MODEL_NAME}_detail_revised.pkl"
)

plt.figure(figsize=(12, 5))
plt.plot(
    backtest_result["trade_time"],
    backtest_result["equity_curve"],
)
plt.title(
    f"{MODEL_NAME} {MODE.title()} — Non-overlapping Incremental PnL"
)
plt.xlabel("Trade time")
plt.ylabel("Equity")
plt.tight_layout()
plt.show()

pd.Series(backtest_metrics)


## Sharpe年化说明

新版先把500ms非重叠净收益聚合为每日净收益，再使用：

\[
Sharpe_{annual}=\sqrt{252}rac{mean(r_{daily})}{std(r_{daily})}
\]

这避免使用固定 `bars_per_day=10000` 对重叠3秒收益进行年化。测试期交易日较少时，Sharpe仍然非常不稳定，必须同时报告交易日数量。


## 当前仍未模拟的实盘约束

- 盘口队列位置与成交概率
- 部分成交与撤单
- 合约乘数、tick size和保证金
- 动态市场冲击
- 网络与模型计算延迟分布
- 涨跌停、临时停牌和异常行情

当前执行价格以调仓快照的ask/bid记录，PnL以mid-price逐步盯市，并显式扣除半价差、手续费和额外滑点。这比原始回测更保守，但仍是研究级近似，不代表实盘成交保证。


## 运行后检查清单

- Validation阈值必须在进入Test前冻结。
- Test预测文件只能在最终模型冻结后生成。
- 反手仓位变化应为2，并承担两倍成本。
- 每个交易时段结束必须回到空仓。
- PnL必须由非重叠500ms收益组成。
- 不保证优化后收益提升。
